In [10]:
# Performance config
import os

CPU_THREADS = min(32, os.cpu_count() or 32)
os.environ["OMP_NUM_THREADS"] = str(CPU_THREADS)
os.environ["MKL_NUM_THREADS"] = str(CPU_THREADS)
os.environ["OPENBLAS_NUM_THREADS"] = str(CPU_THREADS)
os.environ["NUMEXPR_NUM_THREADS"] = str(CPU_THREADS)
os.environ["VECLIB_MAXIMUM_THREADS"] = str(CPU_THREADS)
os.environ["TOKENIZERS_PARALLELISM"] = "false"


# ViLT fine-tune (classic VQA transformer)

Train a single VQA model with a global answer vocabulary.


In [11]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from PIL import Image

import torch
from torch.utils.data import Dataset

from transformers import ViltProcessor, ViltForQuestionAnswering, Trainer, TrainingArguments


In [12]:
from pathlib import Path
import os

def find_imageclef_root() -> Path:
    env_root = os.environ.get("IMAGECLEF_MEDVQA_GI_ROOT")
    if env_root:
        p = Path(env_root).expanduser().resolve()
        if (p / "0_dataset_prep").exists():
            return p
        raise RuntimeError(f"IMAGECLEF_MEDVQA_GI_ROOT set but missing 0_dataset_prep: {p}")

    if "__file__" in globals():
        p = Path(__file__).resolve()
        root = p.parents[2]
        if root.name == "ImageCLEF_MEDVQA_GI_2023" and (root / "0_dataset_prep").exists():
            return root

    cwd = Path.cwd().resolve()
    for p in [cwd] + list(cwd.parents):
        if p.name == "ImageCLEF_MEDVQA_GI_2023" and (p / "0_dataset_prep").exists():
            return p

    raise RuntimeError(
        "Could not locate ImageCLEF_MEDVQA_GI_2023 root. "
        "Run from within the ImageCLEF_MEDVQA_GI_2023 folder or set IMAGECLEF_MEDVQA_GI_ROOT."
    )

ROOT = find_imageclef_root()
sys.path.append(str(ROOT))

from common import (
    find_long_table,
    load_long_table,
    load_label_maps,
    add_label_ids,
    compute_metrics_per_question,
    compute_binary_metrics,
    save_metrics,
    save_predictions,
    normalize_answer,
    OOV_TOKEN,
)

DATA_PATH = find_long_table(ROOT)
LABEL_MAP_DIR = ROOT / "0_dataset_prep" / "out" / "label_maps"
OUT_DIR = ROOT / "2_vqa_models" / "out" / "04_vilt_finetune"
OUT_DIR.mkdir(parents=True, exist_ok=True)

MODEL_NAME = "dandelin/vilt-b32-finetuned-vqa"
BATCH_SIZE = 8
EPOCHS = 3
LR = 5e-5
MAX_TRAIN_SAMPLES = int(os.environ.get("MAX_TRAIN_SAMPLES", "0")) or None
MAX_EVAL_SAMPLES = int(os.environ.get("MAX_EVAL_SAMPLES", "0")) or None
MAX_TEXT_LEN = int(os.environ.get("MAX_TEXT_LEN", "128")) or 128
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"


In [13]:
label_maps = load_label_maps(LABEL_MAP_DIR)
long_df = load_long_table(DATA_PATH)

train_df = long_df[long_df["split"] == "train"].copy()
val_df = long_df[long_df["split"] == "validation"].copy()
test_df = long_df[long_df["split"] == "test"].copy()

if MAX_TRAIN_SAMPLES:
    train_df = train_df.head(MAX_TRAIN_SAMPLES)
if MAX_EVAL_SAMPLES:
    val_df = val_df.head(MAX_EVAL_SAMPLES)
    test_df = test_df.head(MAX_EVAL_SAMPLES)

# Build global answer vocab from train
answers = sorted({normalize_answer(a) for a in train_df["answer_norm"].tolist()})
if OOV_TOKEN not in answers:
    answers.append(OOV_TOKEN)
answer_to_id = {a: i for i, a in enumerate(answers)}
id_to_answer = {i: a for a, i in answer_to_id.items()}


In [14]:
processor = ViltProcessor.from_pretrained(MODEL_NAME)
model = ViltForQuestionAnswering.from_pretrained(
    MODEL_NAME,
    num_labels=len(answer_to_id),
    id2label=id_to_answer,
    label2id=answer_to_id,
    ignore_mismatched_sizes=True,
)
model.to(DEVICE)


Some weights of ViltForQuestionAnswering were not initialized from the model checkpoint at dandelin/vilt-b32-finetuned-vqa and are newly initialized because the shapes did not match:
- classifier.3.weight: found shape torch.Size([3129, 1536]) in the checkpoint and torch.Size([55, 1536]) in the model instantiated
- classifier.3.bias: found shape torch.Size([3129]) in the checkpoint and torch.Size([55]) in the model instantiated
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


ViltForQuestionAnswering(
  (vilt): ViltModel(
    (embeddings): ViltEmbeddings(
      (text_embeddings): TextEmbeddings(
        (word_embeddings): Embedding(30522, 768)
        (position_embeddings): Embedding(40, 768)
        (token_type_embeddings): Embedding(2, 768)
        (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
        (dropout): Dropout(p=0.0, inplace=False)
      )
      (patch_embeddings): ViltPatchEmbeddings(
        (projection): Conv2d(3, 768, kernel_size=(32, 32), stride=(32, 32))
      )
      (token_type_embeddings): Embedding(2, 768)
      (dropout): Dropout(p=0.0, inplace=False)
    )
    (encoder): ViltEncoder(
      (layer): ModuleList(
        (0-11): 12 x ViltLayer(
          (attention): ViltAttention(
            (attention): ViltSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=76

In [15]:
class VQADataset(Dataset):
    def __init__(self, df: pd.DataFrame):
        self.df = df.reset_index(drop=True)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        image = Image.open(row["image_path"]).convert("RGB")
        enc = processor(
            image,
            row["question_text"],
            return_tensors="pt",
            padding="max_length",
            truncation=True,
            max_length=MAX_TEXT_LEN,
        )
        item = {k: v.squeeze(0) for k, v in enc.items()}
        item["labels"] = torch.tensor(answer_to_id.get(normalize_answer(row["answer_norm"]), answer_to_id[OOV_TOKEN]))
        return item

train_ds = VQADataset(train_df)
val_ds = VQADataset(val_df) if len(val_df) else None


def collate_fn(batch):
    return {k: torch.stack([b[k] for b in batch]) for k in batch[0]}


In [16]:
import inspect

args_kwargs = {
    "output_dir": str(OUT_DIR / "checkpoints"),
    "per_device_train_batch_size": BATCH_SIZE,
    "per_device_eval_batch_size": BATCH_SIZE,
    "num_train_epochs": EPOCHS,
    "learning_rate": LR,
    "save_strategy": "epoch",
    "logging_steps": 50,
    "fp16": torch.cuda.is_available(),
    "remove_unused_columns": False,
}

_eval_value = "epoch" if val_ds is not None else "no"
_sig = inspect.signature(TrainingArguments.__init__)
if "evaluation_strategy" in _sig.parameters:
    args_kwargs["evaluation_strategy"] = _eval_value
elif "eval_strategy" in _sig.parameters:
    args_kwargs["eval_strategy"] = _eval_value

args = TrainingArguments(**args_kwargs)


In [17]:
# Trainer
trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    data_collator=collate_fn,
)


In [18]:
# Train
trainer.train()


RuntimeError: stack expects each tensor to be equal size, but got [9] at entry 0 and [13] at entry 1

In [ ]:
# Predict on validation/test
pred_frames = []

for split_name, df_split in [("validation", val_df), ("test", test_df)]:
    if len(df_split) == 0:
        continue
    ds = VQADataset(df_split)
    preds = trainer.predict(ds)
    pred_ids = np.argmax(preds.predictions, axis=1)
    pred_ans = [id_to_answer[int(i)] for i in pred_ids]

    out = df_split.copy()
    out["pred_answer"] = pred_ans
    pred_frames.append(out)

if pred_frames:
    pred_df = pd.concat(pred_frames, ignore_index=True)
else:
    pred_df = pd.DataFrame()


In [ ]:
if len(pred_df):
    pred_df = add_label_ids(pred_df, label_maps, ans_col="answer_norm", out_col="label_id")
    pred_df = add_label_ids(pred_df, label_maps, ans_col="pred_answer", out_col="pred_label_id")

    for split in sorted(pred_df["split"].unique()):
        df_split = pred_df[pred_df["split"] == split]
        overall, per_q = compute_metrics_per_question(df_split, "label_id", "pred_label_id")
        binary = compute_binary_metrics(df_split, label_maps, "label_id", "pred_label_id")
        split_out = OUT_DIR / split
        save_metrics(split_out, overall, per_q, binary)
        save_predictions(
            df_split,
            split_out,
            columns=["image_id", "question_id", "answer_norm", "pred_answer", "split"],
        )

OUT_DIR
